In [96]:
from news_data import load_saved_data
import numpy as np
data = load_saved_data("news_data.pkl")

X_train = data["X_train"]
X_test = data["X_test"]
y_train = data["y_train"]
y_test = data["y_test"]
word_to_idx = data["word_to_idx"]
vocab_size = data["vocab_size"]

删除空样本

In [97]:
new_X_train = []
new_y_train = []

for x, y in zip(X_train, y_train):
    is_empty = False

    if isinstance(x, str):
        if not x.strip():
            is_empty = True
    else:
        arr = np.array(x)
        if len(arr) == 0 or np.all(arr == 0):
            is_empty = True

    if not is_empty:
        new_X_train.append(x)
        new_y_train.append(y)

print("原始样本数:", len(X_train))
print("删除后样本数:", len(new_X_train))
print("删除的空样本数:", len(X_train) - len(new_X_train))

原始样本数: 1079
删除后样本数: 1058
删除的空样本数: 21


In [98]:
X_train=new_X_train
y_train=new_y_train

word2vec

In [99]:
import numpy as np

PAD_IDX = word_to_idx["<PAD>"]
UNK_IDX = word_to_idx["<UNK>"]

# 更推荐：用未padding的X_train
train_corpus = [text.split() for text in X_train if text.strip() != ""]

print(train_corpus[:2])
print("语料句子数:", len(train_corpus))

[['library', 'of', 'congress', 'to', 'host', 'dead', 'sea', 'scroll', 'symposium', 'april', 'to', 'national', 'and', 'assignment', 'desks', 'daybook', 'editor', 'contact', 'john', 'sullivan', 'or', 'lucy', 'suddreth', 'both', 'of', 'the', 'library', 'of', 'congress', 'washington', 'april', 'a', 'symposium', 'on', 'the', 'dead', 'sea', 'scrolls', 'will', 'be', 'held', 'at', 'the', 'library', 'of', 'congress', 'on', 'wednesday', 'april', 'and', 'thursday', 'april', 'the', 'twoday', 'program', 'cosponsored', 'by', 'the', 'library', 'and', 'baltimore', 'hebrew', 'university', 'with', 'additional', 'support', 'from', 'the', 'project', 'judaica', 'foundation', 'will', 'be', 'held', 'in', 'the', 'librarys', 'mumford', 'room', 'sixth', 'floor', 'madison', 'building', 'seating', 'is', 'limited', 'and', 'admission', 'to', 'any', 'session', 'of', 'the', 'symposium', 'must', 'be', 'requested', 'in', 'writing', 'see', 'note', 'a', 'the', 'symposium', 'will', 'be', 'held', 'one', 'week', 'before', '

In [100]:
from gensim.models import Word2Vec

embed_dim = 256

w2v_model = Word2Vec(
    sentences=train_corpus,
    vector_size=embed_dim,
    window=5,
    min_count=1,
    workers=4,
    sg=1,          # 1=skip-gram, 0=CBOW
    epochs=10
)

print("word2vec词表大小:", len(w2v_model.wv))
print("library 是否在词表中:", "library" in w2v_model.wv)

word2vec词表大小: 15822
library 是否在词表中: True


In [101]:
import numpy as np

PAD_IDX = word_to_idx["<PAD>"]
UNK_IDX = word_to_idx["<UNK>"]

embedding_matrix = np.random.normal(
    loc=0.0,
    scale=0.1,
    size=(len(word_to_idx), embed_dim)
).astype(np.float32)

# PAD 设成全 0
embedding_matrix[PAD_IDX] = np.zeros(embed_dim, dtype=np.float32)


# 可选：给 UNK 一个平均向量
valid_vectors = []
for word, idx in word_to_idx.items():
    if word in ["<PAD>", "<UNK>"]:
        continue
    if word in w2v_model.wv:
        valid_vectors.append(w2v_model.wv[word])

if len(valid_vectors) > 0:
    embedding_matrix[UNK_IDX] = np.mean(valid_vectors, axis=0)

print("embedding_matrix.shape =", embedding_matrix.shape)

embedding_matrix.shape = (11753, 256)


1.将 X_train 和 X_test 文本 padding 到相同的长度

In [102]:
train_lengths = [len(seq) for seq in X_train]
test_lengths = [len(seq) for seq in X_test]

print("训练集样本数:", len(X_train))
print("测试集样本数:", len(X_test))
print("训练集最长长度:", max(train_lengths))
print("测试集最长长度:", max(test_lengths))
print("训练集平均长度:", sum(train_lengths) / len(train_lengths))

训练集样本数: 1058
测试集样本数: 717
训练集最长长度: 45731
测试集最长长度: 28696
训练集平均长度: 1283.7003780718337


1.1 padding function

In [103]:
def pad_sequences(sequences, max_len, pad_value=0):
    padded_sequences = []
    
    for seq in sequences:
        seq = list(seq)
        
        if len(seq) < max_len:
            seq = seq + [pad_value] * (max_len - len(seq))
        else:
            seq = seq[:max_len]
        
        padded_sequences.append(seq)
    
    return np.array(padded_sequences, dtype=np.int64)

1.2 translate from text to index

In [104]:
def texts_to_indices(texts, word_to_idx, unk_idx):
    sequences = []
    for text in texts:
        # 如果 text 是一句字符串，就按空格切分
        tokens = text.split()
        
        # 转成索引，不在词表里的词用 UNK
        seq = [word_to_idx.get(token, unk_idx) for token in tokens]
        sequences.append(seq)
    
    return sequences

In [105]:
PAD_IDX = word_to_idx["<PAD>"]
UNK_IDX = word_to_idx["<UNK>"]

In [106]:
X_train_idx = texts_to_indices(X_train, word_to_idx, UNK_IDX)
X_test_idx  = texts_to_indices(X_test, word_to_idx, UNK_IDX)

In [107]:
max_len = int(np.percentile(train_lengths, 99.5))
print("max_len =", max_len)

max_len = 12687


In [108]:
X_train_pad = pad_sequences(X_train_idx, max_len=max_len, pad_value=PAD_IDX)
X_test_pad  = pad_sequences(X_test_idx, max_len=max_len, pad_value=PAD_IDX)

In [109]:
print("X_train_pad.shape =", X_train_pad.shape)
print("X_test_pad.shape =", X_test_pad.shape)

X_train_pad.shape = (1058, 12687)
X_test_pad.shape = (717, 12687)


In [110]:
import numpy as np
y_train = np.array(y_train)
y_test = np.array(y_test)

print(y_train.shape, y_test.shape)

(1058,) (717,)


Bidirectional LSTM

In [111]:
import torch
import torch.nn as nn
PAD_IDX = word_to_idx["<PAD>"]

X_train_tensor = torch.tensor(X_train_pad, dtype=torch.long)
X_test_tensor  = torch.tensor(X_test_pad, dtype=torch.long)

y_train_tensor = torch.tensor(y_train, dtype=torch.long)
y_test_tensor  = torch.tensor(y_test, dtype=torch.long)

train_lengths = (X_train_tensor != PAD_IDX).sum(dim=1)
test_lengths  = (X_test_tensor != PAD_IDX).sum(dim=1)

In [112]:
train_mask = train_lengths > 0
test_mask = test_lengths > 0

X_train_tensor = X_train_tensor[train_mask]
y_train_tensor = y_train_tensor[train_mask]
train_lengths = train_lengths[train_mask]

X_test_tensor = X_test_tensor[test_mask]
y_test_tensor = y_test_tensor[test_mask]
test_lengths = test_lengths[test_mask]

In [113]:
from torch.utils.data import TensorDataset, DataLoader

train_dataset = TensorDataset(X_train_tensor, train_lengths, y_train_tensor)
test_dataset  = TensorDataset(X_test_tensor, test_lengths, y_test_tensor)

batch_size = 64
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader  = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [114]:
import torch
import torch.nn as nn

class BiLSTMClassifier(nn.Module):
    def __init__(
        self,
        vocab_size,
        embed_dim,
        hidden_dim,
        pad_idx,
        num_classes=2,
        pretrained_embeddings=None,
        freeze_embedding=False
    ):
        super().__init__()

        if pretrained_embeddings is not None:
            self.embedding = nn.Embedding.from_pretrained(
                embeddings=torch.tensor(pretrained_embeddings, dtype=torch.float32),
                freeze=freeze_embedding,
                padding_idx=pad_idx
            )
        else:
            self.embedding = nn.Embedding(
                num_embeddings=vocab_size,
                embedding_dim=embed_dim,
                padding_idx=pad_idx
            )

        self.lstm = nn.LSTM(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            num_layers=1,
            batch_first=True,
            bidirectional=True
        )

        self.fc = nn.Linear(hidden_dim * 2, num_classes)

        self.init_forget_gate_bias(value=1.0)

    def init_forget_gate_bias(self, value=1.0):
        hidden_dim = self.lstm.hidden_size
        for name, param in self.lstm.named_parameters():
            if "bias" in name:
                param.data[hidden_dim:2 * hidden_dim].fill_(value)

    def forward(self, x, lengths):
        x_embed = self.embedding(x)

        packed = nn.utils.rnn.pack_padded_sequence(
            x_embed, lengths.cpu(), batch_first=True, enforce_sorted=False
        )

        _, (h_n, c_n) = self.lstm(packed)

        h_forward = h_n[0]
        h_backward = h_n[1]

        h_concat = torch.cat([h_forward, h_backward], dim=1)
        logits = self.fc(h_concat)

        return logits

In [115]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = BiLSTMClassifier(
    vocab_size=len(word_to_idx),
    embed_dim=embed_dim,
    hidden_dim=256,
    pad_idx=PAD_IDX,
    num_classes=2,
    pretrained_embeddings=embedding_matrix,
    freeze_embedding=False   # False: 训练时继续微调
).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

In [116]:
from sklearn.metrics import accuracy_score

def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    total_loss = 0
    all_preds = []
    all_labels = []

    for X_batch, lengths_batch, y_batch in dataloader:
        X_batch = X_batch.to(device)
        lengths_batch = lengths_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()

        logits = model(X_batch, lengths_batch)   # [B, 2]
        loss = criterion(logits, y_batch)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        preds = torch.argmax(logits, dim=1)
        all_preds.extend(preds.detach().cpu().numpy())
        all_labels.extend(y_batch.detach().cpu().numpy())

    return total_loss / len(dataloader), accuracy_score(all_labels, all_preds)

In [117]:
def evaluate(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for X_batch, lengths_batch, y_batch in dataloader:
            X_batch = X_batch.to(device)
            lengths_batch = lengths_batch.to(device)
            y_batch = y_batch.to(device)

            logits = model(X_batch, lengths_batch)
            loss = criterion(logits, y_batch)

            total_loss += loss.item()

            preds = torch.argmax(logits, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(y_batch.cpu().numpy())

    return total_loss / len(dataloader), accuracy_score(all_labels, all_preds), all_preds, all_labels

In [118]:
epochs = 20

for epoch in range(epochs):
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    test_loss, test_acc, _, _ = evaluate(model, test_loader, criterion, device)

    print(f"Epoch [{epoch+1}/{epochs}] "
          f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f} | "
          f"Test Loss: {test_loss:.4f}, Test Acc: {test_acc:.4f}")

Epoch [1/20] Train Loss: 0.6939, Train Acc: 0.5350 | Test Loss: 0.6695, Test Acc: 0.5971
Epoch [2/20] Train Loss: 0.6578, Train Acc: 0.6144 | Test Loss: 0.6686, Test Acc: 0.6345
Epoch [3/20] Train Loss: 0.6241, Train Acc: 0.7004 | Test Loss: 0.6401, Test Acc: 0.6173
Epoch [4/20] Train Loss: 0.4922, Train Acc: 0.7722 | Test Loss: 0.6177, Test Acc: 0.6676
Epoch [5/20] Train Loss: 0.3108, Train Acc: 0.8941 | Test Loss: 0.9806, Test Acc: 0.6647
Epoch [6/20] Train Loss: 0.2539, Train Acc: 0.9253 | Test Loss: 0.7293, Test Acc: 0.6590
Epoch [7/20] Train Loss: 0.3137, Train Acc: 0.8658 | Test Loss: 0.5572, Test Acc: 0.7223
Epoch [8/20] Train Loss: 0.1268, Train Acc: 0.9707 | Test Loss: 0.5668, Test Acc: 0.7856
Epoch [9/20] Train Loss: 0.0802, Train Acc: 0.9716 | Test Loss: 1.1670, Test Acc: 0.6489
Epoch [10/20] Train Loss: 0.1745, Train Acc: 0.9244 | Test Loss: 0.6443, Test Acc: 0.7237
Epoch [11/20] Train Loss: 0.0444, Train Acc: 0.9839 | Test Loss: 0.7187, Test Acc: 0.7194
Epoch [12/20] Train

In [119]:
from sklearn.metrics import classification_report

test_loss, test_acc, y_pred, y_true = evaluate(model, test_loader, criterion, device)

print("Test Loss:", test_loss)
print("Test Accuracy:", test_acc)
print(classification_report(y_true, y_pred, digits=4))

Test Loss: 0.9252303676171736
Test Accuracy: 0.7741007194244605
              precision    recall  f1-score   support

           0     0.7584    0.7267    0.7422       311
           1     0.7859    0.8125    0.7990       384

    accuracy                         0.7741       695
   macro avg     0.7721    0.7696    0.7706       695
weighted avg     0.7736    0.7741    0.7736       695



In [120]:
save_path = "bilstm_text_classifier.pth"

checkpoint = {
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "word_to_idx": word_to_idx,
    "vocab_size": len(word_to_idx),
    "embed_dim": 512,
    "hidden_dim": 512,
    "pad_idx": PAD_IDX,
    "num_classes": 2,
    "max_len": X_train_pad.shape[1],
    "test_acc": test_acc
}

torch.save(checkpoint, save_path)
print(f"模型已保存到: {save_path}")

模型已保存到: bilstm_text_classifier.pth
